# One-vs-One Logistic Regression

# Cybersecurity Multiclass Classification (Synthetic, Linearly Separable)

This notebook loads the provided dataset and trains a **logistic regression** classifier using a specific multiclass strategy.

**Dataset file:** `cyber_multiclass_linear.csv`

Classes:
- benign
- credential_stuffing
- port_scan
- ddos
- malware_c2
- data_exfiltration


## Strategy
This uses **One-vs-One (OvO)**: one binary classifier per pair of classes; predictions are combined by voting.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

import matplotlib.pyplot as plt

# Path to the CSV in this workspace
CSV_PATH = "/mnt/data/cyber_multiclass_linear.csv"

# 1) Load data
df = pd.read_csv(CSV_PATH)
display(df.head())

# 2) Split features/labels
feature_cols = [c for c in df.columns if c != "label"]
X = df[feature_cols].values
y = df["label"].values

# 3) Train/test split (stratified for balanced classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Classes:", np.unique(y))


In [ ]:
# --- Visualization: how separable are the classes? ---
# We'll visualize the feature space in a few ways:
# 1) 2D PCA projection (all features)
# 2) A few hand-picked 2D feature plots

from sklearn.decomposition import PCA

# Color mapping for consistent plots
classes = np.unique(y)
colors = plt.cm.get_cmap("tab10", len(classes))
color_map = {cls: colors(i) for i, cls in enumerate(classes)}

def scatter_2d(x2d, y_labels, title, xlabel, ylabel):
    plt.figure(figsize=(8, 6))
    for cls in classes:
        mask = (y_labels == cls)
        plt.scatter(x2d[mask, 0], x2d[mask, 1], s=12, alpha=0.85, label=cls, c=[color_map[cls]])
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

# 1) PCA (2D)
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)
scatter_2d(
    X_2d, y,
    title="PCA (2D) projection of all features",
    xlabel=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)",
    ylabel=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)"
)

# 2) Hand-picked 2D feature views (often very separable for this synthetic dataset)
def plot_feature_pair(f1, f2):
    i1, i2 = feature_cols.index(f1), feature_cols.index(f2)
    x2d = np.c_[X[:, i1], X[:, i2]]
    scatter_2d(x2d, y, title=f"Feature space: {f1} vs {f2}", xlabel=f1, ylabel=f2)

plot_feature_pair("failed_login_rate", "login_attempts")
plot_feature_pair("unique_dst_ports", "connection_rate")
plot_feature_pair("process_tree_anomaly", "url_reputation_score")
plot_feature_pair("outbound_bytes_kb", "inbound_bytes_kb")


In [ ]:
# --- Model: One-vs-One (OvO) with Logistic Regression base learner ---
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsOneClassifier
from sklearn.decomposition import PCA

base_lr = LogisticRegression(
    solver="lbfgs",
    max_iter=2000,
    random_state=42
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", OneVsOneClassifier(base_lr))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)
print("\nClassification report:\n", classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, xticks_rotation=45, cmap="Blues")
plt.title("Confusion Matrix — One-vs-One Logistic Regression")
plt.tight_layout()
plt.show()

# Visualize OvO decision regions in 2D PCA (approximate)
pca2 = PCA(n_components=2, random_state=42)
X_train_2d = pca2.fit_transform(X_train)

viz_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", OneVsOneClassifier(LogisticRegression(solver="lbfgs", max_iter=2000, random_state=42)))
])
viz_model.fit(X_train_2d, y_train)

x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 350), np.linspace(y_min, y_max, 350))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = viz_model.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, pd.Categorical(Z, categories=classes).codes, alpha=0.25)

for cls in classes:
    m = (y_train == cls)
    plt.scatter(X_train_2d[m, 0], X_train_2d[m, 1], s=10, alpha=0.8, label=cls, c=[color_map[cls]])

plt.title("Approx. decision regions (trained on 2D PCA) — OvO Logistic Regression")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
